# 🏆 Notebook 07: BERTized Agentic Context Engineering (ACE) Pipeline
### High-Precision Lifestyle Factor (LSF) Extraction in a Post-BERT Proposer Setting
#### Target Dataset: Combined 200Abstracts Train (120) + Val (40) = 160 Abstract Pairs

---

## 📌 Architectural Overview

This notebook implements the **BERTized Agentic Context Engineering (ACE)** pipeline. Rather than forcing generative LLMs to scan full raw abstracts from scratch (which leads to severe hallucination and boundary instability), the system operates in a **two-phase decoupled architecture**:

```
                                  ┌─────────────────────────────────────────────────────────────┐
                                  │             PHASE 1: SPANNER NEURAL PROPOSER                │
  Raw Abstract Text ─────────────►│  • Proposes candidate spans (start_char, end_char)          │
                                  │  • Computes top-1, top-2, uncertainty, novelty, and margin  │
                                  └──────────────────────────────┬──────────────────────────────┘
                                                                 │ Candidate Proposals + BERT Evidence
                                                                 ▼
┌────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                           PHASE 2: 3-AGENT COLLABORATIVE ACE CURATION LOOP                                     │
│                                                                                                                │
│   ┌───────────────────────────┐      ┌───────────────────────────┐      ┌───────────────────────────┐          │
│   │   1. ACE GENERATOR (v2)   │      │   2. ACE REFLECTOR (v2)   │      │    3. ACE CURATOR (v2)    │          │
│   │  • Reads BERT evidence    │      │  • Evaluates in-scope     │      │  • Screens against BASE   │          │
│   │  • Classifies candidate   │─────►│    BERT candidate errors  │─────►│    RULES & 4 Genericity   │          │
│   │    spans (9 LSF, Out, O)  │      │  • Checks evidence anchor │      │    Tests                  │          │
│   │  • Cites dynamicGuideline │      │  • Formulates key insight │      │  • Formulates ADD/MODIFY  │          │
│   └───────────────────────────┘      └───────────────────────────┘      └─────────────┬─────────────┘          │
│                 ▲                                                                     │                        │
│                 │                                                                     ▼                        │
│                 │                    ┌───────────────────────────────────┐     Guideline Operations    │
│                 └────────────────────│    DYNAMIC GUIDELINES MANAGER     │◄────────────────────────────┘       │
│                  Current Playbook    │  • Persistent vector store (Chroma│                                     │
│                                      │  • Auto pruning & helpful/harmful │                                     │
│                                      └───────────────────────────────────┘                                     │
└────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
```

### 1. The 3 Specialized ACE Agents (v2):
1. **Generator (`ACE_Generator_v2.txt`)**: Operates exclusively on candidate spans proposed by SpanNER. Weighs `predicted_label` vs `second_best_label` based on `margin`, `uncertainty`, and `novelty_score`. Applies Base Rules and dynamic guidelines. Max tokens: **16k**.
2. **Reflector (`ACE_Reflector_v2.txt`)**: Diagnostic analyst covering **ONLY in-scope BERT-proposed candidate spans**. Evaluates `Comparison_Label` (`TP`, `FP`, `FN`, `CLSS_ERR`), detects evidence anchoring, identifies root cause, and derives generalizable key insights.
3. **Curator (`ACE_Curator_v2.txt`)**: Senior policy architect acting as the last line of defense. Screens insights against Base Rules and 4 strict genericity tests (Pattern, Necessity, Restatement, Stability) before generating structured `ADD` or `MODIFY` operations.

### 2. Dual-View Standardized Evaluation:
* **View 1 (Candidate-Level Arbitration)**: Evaluates decision accuracy on candidate mentions proposed by SpanNER (`TP / (TP + FN)` across candidate mentions).
* **View 2 (Global End-to-End Dataset Recovery)**: Evaluates total recovery against gold entities in the dataset across standard academic formulations (**Strict CoNLL**, **Partial MUC-7**, and **Candidate Mention**).


In [ ]:
# ==============================================================================
# [CELL 1]: Environment Setup, Paths & Imports
# ==============================================================================
import os
import sys
import json
import time
import shutil
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
from dotenv import load_dotenv

# Robustly resolve PROJECT_ROOT whether running from /notebooks or project root
if os.path.exists("data") and os.path.exists("src"):
    PROJECT_ROOT = os.path.abspath(".")
elif os.path.exists("../data") and os.path.exists("../src"):
    PROJECT_ROOT = os.path.abspath("..")
else:
    PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Explicitly load .env configuration
dotenv_path = os.path.join(PROJECT_ROOT, ".env")
load_dotenv(dotenv_path, override=True)

import torch
import numpy as np

# Core Pipeline & Architecture Imports
from src.architectures.hybrid_linkner.pipeline import LinkNERPipeline
from src.architectures.agentic_guidelines.manager import DynamicGuidelinesManager
from src.architectures.agentic_guidelines.bertized_ace import (
    BertizedGenerator,
    BertizedReflector,
    BertizedCurator,
    BertizedACEPipeline
)
from src.common.audit_logger import (
    classify_8_outcome_str,
    match_gt_label,
    create_run_directory
)
from src.common.label_mapping import canonicalize_label
from src.common.evaluation_framework import (
    evaluate_doc_4_formulations,
    aggregate_4_formulations,
    format_4_formulations_summary_box
)

print(f"✅ Environment initialized. Project Root: {PROJECT_ROOT}")
print(f"⚡ Device: {'CUDA' if torch.cuda.is_available() else ('MPS' if torch.backends.mps.is_available() else 'CPU')}")


In [ ]:
# ==============================================================================
# [CELL 2]: Experiment Configuration (160 Abstracts & 16k Max Tokens)
# ==============================================================================
SEED = int(os.getenv("RANDOM_SEED", "42"))
torch.manual_seed(SEED)
np.random.seed(SEED)

# 1. Target Dataset: Combined 200Abstracts/train (120) + 200Abstracts/val (40) = 160 Abstract Pairs
DATASET_DIRS = [
    os.path.join(PROJECT_ROOT, "data/new/200Abstracts/train"),
    os.path.join(PROJECT_ROOT, "data/new/200Abstracts/val")
]

# 2. Output & Guideline Storage
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "output/bertized_ace")
DYNAMIC_GUIDELINES_PATH = os.path.join(OUTPUT_DIR, "dynamicGuidelines.json")
CHROMA_DIR = os.path.join(OUTPUT_DIR, "chroma_db")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHROMA_DIR, exist_ok=True)

# 3. Model & LLM Provider Configuration (.env driven: 9Router / gemini-cli, 16k Max Tokens)
MODEL_NAME = os.getenv("LLM_MODEL", "gemini-cli")
BACKEND = os.getenv("LLM_BACKEND", "api")
PROVIDER = os.getenv("LLM_PROVIDER", "9router")
API_BASE_URL = os.getenv("LLM_API_BASE_URL", os.getenv("API_Base_URL", "http://localhost:20128/v1"))
MAX_TOKENS = 16384  # 16k Max Tokens for deep reflection & guideline reasoning

# 4. SpanNER Neural Proposer Configuration
SPANNER_CHECKPOINT = os.path.join(PROJECT_ROOT, "models/SpanNER_LSF/best_spanner_160train.pt")
USE_NEW_LABELS = True
BATCH_SIZE = 5
USE_NMS = False

print("=" * 80)
print("📋 BERTIZED ACE MASTER CONFIGURATION")
print("=" * 80)
print(f"  • Target Dataset Folders : {DATASET_DIRS}")
print(f"  • Expected Total Abstracts: 160 abstract pairs (120 train + 40 val)")
print(f"  • SpanNER Model Checkpoint: {SPANNER_CHECKPOINT}")
print(f"  • LLM Provider / Gateway : {PROVIDER.upper()} -> {API_BASE_URL}")
print(f"  • Active Model Identifier : {MODEL_NAME}")
print(f"  • Max Tokens for ACE Pipeline: {MAX_TOKENS} (16k)")
print(f"  • Dynamic Guidelines JSON: {DYNAMIC_GUIDELINES_PATH}")
print(f"  • ChromaDB Vector Store  : {CHROMA_DIR}")
print(f"  • Target Label Schema    : {'NEW (Relabeled)' if USE_NEW_LABELS else 'LEGACY'}")
print("=" * 80)


In [ ]:
# ==============================================================================
# [CELL 3]: Load Combined 160 Abstracts Dataset (.txt + .ann pairs)
# Robust parser handles fragmented/discontinuous BRAT spans with semicolons
# ==============================================================================
def load_combined_abstract_pairs(dataset_dirs):
    all_pairs = []
    seen_doc_ids = set()
    for d_dir in dataset_dirs:
        if not os.path.exists(d_dir):
            print(f"⚠️ Directory not found: {d_dir}")
            continue
        txt_files = sorted([f for f in os.listdir(d_dir) if f.endswith(".txt")])
        for txt_file in txt_files:
            base = os.path.splitext(txt_file)[0]
            if base in seen_doc_ids:
                continue
            txt_path = os.path.join(d_dir, f"{base}.txt")
            ann_path = os.path.join(d_dir, f"{base}.ann")
            
            gt_ents = []
            if os.path.exists(ann_path):
                with open(ann_path, "r", encoding="utf-8") as f:
                    for line in f:
                        line = line.strip()
                        if not line or not line.startswith("T"):
                            continue
                        parts = line.split("	")
                        if len(parts) < 3:
                            continue
                        type_offsets = parts[1]
                        ent_text = parts[2]
                        type_parts = type_offsets.split()
                        lbl = type_parts[0]
                        # Robustly parse offsets, supporting fragmented spans with semicolons (e.g. '811 820;837 855')
                        raw_offsets = type_offsets[len(lbl):].strip().replace(";", " ").split()
                        if len(raw_offsets) >= 2:
                            st_c = int(raw_offsets[0])
                            end_c = int(raw_offsets[-1])
                            gt_ents.append({
                                "label": canonicalize_label(lbl, use_new_labels=USE_NEW_LABELS),
                                "start_char": st_c,
                                "end_char": end_c,
                                "text": ent_text
                            })
            with open(txt_path, "r", encoding="utf-8") as f:
                text = f.read()
            all_pairs.append((txt_path, ann_path, text, gt_ents, base))
            seen_doc_ids.add(base)
    return all_pairs

dataset_abstracts = load_combined_abstract_pairs(DATASET_DIRS)
print(f"📦 Successfully loaded {len(dataset_abstracts)} abstract pairs across {len(DATASET_DIRS)} folders.")

# Partition into Training Curation Split (120 abstracts) and Validation Testing Split (40 abstracts)
train_files = dataset_abstracts[:120]
test_files = dataset_abstracts[120:]

print(f"  • Training / Curation Split : {len(train_files)} abstracts")
print(f"  • Validation / Test Split   : {len(test_files)} abstracts")


In [ ]:
# ==============================================================================
# [CELL 4]: Phase 1 - Fast Neural Batch Candidate Proposer (SpanNER + LOF)
# ==============================================================================
print("=" * 80)
print("🚀 Initializing SpanNER Neural Proposer Pipeline...")
print("=" * 80)

spanner_pipeline = LinkNERPipeline(
    spanner_checkpoint_path=SPANNER_CHECKPOINT,
    use_new_labels=USE_NEW_LABELS,
    use_nms=USE_NMS
)

def extract_candidates_for_split(files_list, split_name="Split"):
    t0 = time.time()
    texts = [item[2] for item in files_list]
    print(f"⚡ Running SpanNER neural forward passes on {len(texts)} {split_name} abstracts (B={BATCH_SIZE})...")
    # Setting threshold to 1.01 retrieves pure SpanNER candidates without invoking LLM
    batch_raw_preds = spanner_pipeline.predict_batch(
        texts=texts,
        batch_size=BATCH_SIZE,
        uncertainty_threshold=1.01,
        use_nms=USE_NMS,
        verbose=False
    )
    
    extracted_records = {}
    total_spans = 0
    for idx, (txt_p, ann_p, text, gt_ents, doc_id) in enumerate(files_list):
        raw_cands = batch_raw_preds[idx]
        processed_cands = []
        for c in raw_cands:
            s_text = c["span_text"]
            st_c = c["start_char"]
            end_c = c["end_char"]
            p_lbl = canonicalize_label(c.get("predicted_label", c.get("spanner_label", "O")), use_new_labels=USE_NEW_LABELS)
            s_lbl = canonicalize_label(c.get("second_best_label", "O"), use_new_labels=USE_NEW_LABELS)
            unc = float(c.get("uncertainty", 0.0))
            nov = float(c.get("novelty", 0.0))
            margin = float(c.get("margin", 0.0))
            gt_lbl = match_gt_label(s_text, st_c, end_c, gt_ents)
            
            cand_dict = {
                "span_text": s_text,
                "entity": s_text,
                "start_char": st_c,
                "end_char": end_c,
                "predicted_label": p_lbl,
                "spanner_label": p_lbl,
                "second_best_label": s_lbl,
                "uncertainty": unc,
                "novelty_score": nov,
                "margin": margin,
                "prob": float(c.get("prob", 0.0)),
                "gt_label": gt_lbl
            }
            processed_cands.append(cand_dict)
            total_spans += 1
        extracted_records[doc_id] = {
            "text": text,
            "candidates": processed_cands,
            "gt_ents": gt_ents
        }
    elapsed = time.time() - t0
    print(f"✅ Extracted {total_spans} candidate spans across {len(files_list)} abstracts in {elapsed:.2f}s ({elapsed/max(1, len(files_list)):.2f}s / abstract).")
    return extracted_records

train_candidates_cache = extract_candidates_for_split(train_files, split_name="Train (120)")
test_candidates_cache = extract_candidates_for_split(test_files, split_name="Test (40)")


In [ ]:
# ==============================================================================
# [CELL 5]: Initialize BERTized ACE Pipeline & Dynamic Guidelines Store
# ==============================================================================
print("=" * 80)
print("🏛️ Initializing DynamicGuidelinesManager (ChromaDB + JSON) & BERTized ACE Pipeline...")
print("=" * 80)

ace_pipeline = BertizedACEPipeline(
    model_name=MODEL_NAME,
    backend=BACKEND,
    dynamicGuidelines_path=DYNAMIC_GUIDELINES_PATH,
    chroma_persist_directory=CHROMA_DIR,
    spanner_pipeline=spanner_pipeline,
    max_tokens=MAX_TOKENS
)

initial_guidelines = ace_pipeline.manager.get_dynamicGuidelines()
initial_bullet_count = sum(len(bullets) for bullets in initial_guidelines.values())
print(f"📚 Initial Guidelines Loaded: {initial_bullet_count} active bullets across {len(initial_guidelines)} sections.")


In [ ]:
# ==============================================================================
# [CELL 6]: Phase 2 - Training & Iterative Guideline Curation (ACE Loop)
# ==============================================================================
# Choose how many training abstracts to run in this curation session (e.g. 20, 50, or all 120):
CURATION_ABSTRACT_LIMIT = int(os.getenv("CURATION_LIMIT", "20"))
curation_subset = train_files[:CURATION_ABSTRACT_LIMIT]

print("=" * 96)
print(f"🚀 Starting BERTized ACE Curation Loop across {len(curation_subset)} training abstracts...")
print("=" * 96)

train_logs = []
step_start = time.time()

for idx, (txt_p, ann_p, text, gt_ents, doc_id) in enumerate(curation_subset, start=1):
    doc_data = train_candidates_cache[doc_id]
    cands = doc_data["candidates"]
    
    print(f"\n📄 [Train Abstract {idx}/{len(curation_subset)}]: {doc_id} ({len(cands)} candidate spans)")
    
    predicted, reflection, curator_output = ace_pipeline.train_abstract(
        abstract=text,
        candidate_spans=cands,
        file_name=doc_id,
        save_outputs_dir=os.path.join(OUTPUT_DIR, "train_agent_logs")
    )
    
    ops = curator_output.get("operations", [])
    used_b = predicted.get("bullet_ids", [])
    key_ins = reflection.get("key_insight", "None")
    
    print(f"   ⚡ Generator Classified: {len(predicted.get('final_answer', []))} spans (Used bullets: {used_b or 'Base Rules'})")
    print(f"   🧠 Reflector Insight  : {key_ins[:90] if key_ins else 'None'}...")
    if ops:
        for op in ops:
            op_type = op.get("type", "ADD")
            sec = op.get("section", op.get("bullet_id", ""))
            c_snip = op.get("content", "")[:70]
            print(f"   🏛️ Curator Action      : [{op_type}] -> {sec} | Content: '{c_snip}...'")
    else:
        print(f"   🛡️ Curator Action      : [Safe-Fail / No Update]")
        
    train_logs.append({
        "doc_id": doc_id,
        "spans_count": len(cands),
        "bullet_ids_used": used_b,
        "key_insight": key_ins,
        "operations": ops
    })

ace_pipeline.save_guidelines()
total_train_time = time.time() - step_start
updated_guidelines = ace_pipeline.manager.get_dynamicGuidelines()
final_bullet_count = sum(len(bullets) for bullets in updated_guidelines.values())

print("\n" + "=" * 96)
print(f"🏁 ACE Training Complete in {total_train_time:.2f}s!")
print(f"📚 Dynamic Guidelines Playbook Evolved: {initial_bullet_count} -> {final_bullet_count} active bullets.")
print("=" * 96)


In [ ]:
# ==============================================================================
# [CELL 7]: Export Curated Guidelines to Markdown & CSV & Snapshot Backup
# ==============================================================================
md_export_path = os.path.join(OUTPUT_DIR, "curated_dynamicGuidelines.md")
csv_export_path = os.path.join(OUTPUT_DIR, "curated_dynamicGuidelines.csv")

# 1. Export Markdown Guidelines
ace_pipeline.export_guidelines_markdown(md_export_path)
print(f"📄 Exported clean Markdown guidelines to: {md_export_path}")

# 2. Export Tabular CSV with Metrics
ace_pipeline.export_guidelines_csv(csv_export_path)
print(f"📊 Exported tabular CSV guidelines to   : {csv_export_path}")

# 3. CLI Management Stats
!python scripts/manage_guidelines.py --guidelines-path {DYNAMIC_GUIDELINES_PATH} --stats


In [ ]:
# ==============================================================================
# [CELL 8]: Phase 3 - Testing & Inference on Unseen Abstracts (Curated Playbook)
# ==============================================================================
# Choose how many test abstracts to evaluate (e.g. 10, 20, or all 40):
TEST_EVAL_LIMIT = int(os.getenv("TEST_LIMIT", "20"))
test_subset = test_files[:TEST_EVAL_LIMIT]

print("=" * 96)
print(f"🔍 Running Generator Inference with Curated Guidelines on {len(test_subset)} test abstracts...")
print("=" * 96)

all_test_preds = []
all_test_gts = [item[3] for item in test_subset]
spanner_baseline_test_preds = []

t0 = time.time()
for idx, (txt_p, ann_p, text, gt_ents, doc_id) in enumerate(test_subset, start=1):
    doc_data = test_candidates_cache[doc_id]
    cands = doc_data["candidates"]
    
    # Run ACE Generator with frozen curated guidelines
    preds = ace_pipeline.predict_abstract(
        abstract=text,
        candidate_spans=cands,
        file_name=doc_id,
        save_outputs_dir=os.path.join(OUTPUT_DIR, "test_agent_logs")
    )
    all_test_preds.append(preds)
    
    # Collect SpanNER proposer baseline predictions for direct comparison
    doc_spanner = []
    for c in cands:
        s_lbl = c["predicted_label"]
        if s_lbl and s_lbl.lower() != "o":
            doc_spanner.append({
                "span_text": c["span_text"],
                "start_char": c["start_char"],
                "end_char": c["end_char"],
                "label": s_lbl
            })
    spanner_baseline_test_preds.append(doc_spanner)
    
    num_pos = sum(1 for p in preds if p["label"] != "O")
    num_pruned = len(preds) - num_pos
    print(f"  • Abstract [{idx}/{len(test_subset)}]: {doc_id} -> {len(cands)} candidates ({num_pos} accepted, {num_pruned} rejected to 'O')")

test_elapsed = time.time() - t0
print(f"\n✅ Inference complete in {test_elapsed:.2f}s ({test_elapsed/max(1, len(test_subset)):.2f}s / abstract).")


## 🏆 Standardized Multi-Tier Evaluation Framework: Dual-View Benchmark

This evaluation framework decouples the performance of the **two architectural stages**:

### 1. View 1: Candidate-Level Arbitration Benchmark (Master Report / Sweep Engine Formulation)
- **Evaluates**: The decision accuracy of the ACE Generator on candidate spans proposed by SpanNER.
- **Recall Denominator**: Evaluated candidate spans matching ground truth (`TP / (TP + FN)` across candidate mentions).
- **Baseline Comparison**: Directly measures whether ACE Curation improved SpanNER's proposed candidate pool.

### 2. View 2: Global End-to-End Dataset Benchmark (Standard Academic Gold Set Recovery)
- **Evaluates**: Total ground truth entity recovery against all gold entities in the test set across standard academic formulations:
  - **Strict CoNLL (1)**: Exact character boundary + exact category match.
  - **Partial MUC-7 (3)**: Any character boundary overlap + exact category match (1-to-1 bipartite lockout).
  - **Candidate Mention (4)**: Candidate mention coverage without 1-to-1 lockout against all gold entities.


In [ ]:
# ==============================================================================
# [CELL 10]: Phase 4 - Standardized Dual-View Evaluation Benchmark
# View 1: Candidate Arbitration Benchmark (Decision Accuracy on Proposed Mentions)
# View 2: Global End-to-End Dataset Benchmark (Standard Academic Gold Set Recovery)
# ==============================================================================
# ------------------------------------------------------------------------------
# Build Flat Audit Records across all test abstracts
# ------------------------------------------------------------------------------
all_audit_records = []
all_outcomes = []

for doc_idx, (txt_p, ann_p, text, gt_ents, doc_id) in enumerate(test_subset):
    preds = all_test_preds[doc_idx]
    for p in preds:
        gt_lbl = match_gt_label(p["span_text"], p["start_char"], p["end_char"], gt_ents)
        s_lbl = canonicalize_label(p.get("spanner_label", "O"), use_new_labels=USE_NEW_LABELS)
        f_lbl = canonicalize_label(p.get("label", "O"), use_new_labels=USE_NEW_LABELS)
        
        all_audit_records.append({
            "span": p["span_text"],
            "spanner_label": s_lbl,
            "hybrid_label": f_lbl,
            "ground_truth": gt_lbl
        })
        outcome_str = classify_8_outcome_str(gt_label=gt_lbl, spanner_label=s_lbl, final_label=f_lbl, escalated=True)
        all_outcomes.append(outcome_str)

# ------------------------------------------------------------------------------
# 1. VIEW 1: Candidate-Level Arbitration Benchmark
# ------------------------------------------------------------------------------
classes = sorted(list(set(r["ground_truth"] for r in all_audit_records if r["ground_truth"] != "O") |
                      set(r["hybrid_label"] for r in all_audit_records if r["hybrid_label"] != "O") |
                      set(r["spanner_label"] for r in all_audit_records if r["spanner_label"] != "O")))

per_class_table = []
tot_s_tp = tot_s_fp = tot_s_fn = 0
tot_h_tp = tot_h_fp = tot_h_fn = 0
s_f1_list, h_f1_list = [], []

for cls in classes:
    s_tp = sum(1 for r in all_audit_records if r["spanner_label"] == cls and r["ground_truth"] == cls)
    s_fp = sum(1 for r in all_audit_records if r["spanner_label"] == cls and r["ground_truth"] != cls)
    s_fn = sum(1 for r in all_audit_records if r["ground_truth"] == cls and r["spanner_label"] != cls)
    
    h_tp = sum(1 for r in all_audit_records if r["hybrid_label"] == cls and r["ground_truth"] == cls)
    h_fp = sum(1 for r in all_audit_records if r["hybrid_label"] == cls and r["ground_truth"] != cls)
    h_fn = sum(1 for r in all_audit_records if r["ground_truth"] == cls and r["hybrid_label"] != cls)
    
    s_p = s_tp / (s_tp + s_fp) if (s_tp + s_fp) > 0 else 0.0
    s_r = s_tp / (s_tp + s_fn) if (s_tp + s_fn) > 0 else 0.0
    s_f1 = (2 * s_p * s_r) / (s_p + s_r) if (s_p + s_r) > 0 else 0.0
    
    h_p = h_tp / (h_tp + h_fp) if (h_tp + h_fp) > 0 else 0.0
    h_r = h_tp / (h_tp + h_fn) if (h_tp + h_fn) > 0 else 0.0
    h_f1 = (2 * h_p * h_r) / (h_p + h_r) if (h_p + h_r) > 0 else 0.0
    
    tot_s_tp += s_tp; tot_s_fp += s_fp; tot_s_fn += s_fn
    tot_h_tp += h_tp; tot_h_fp += h_fp; tot_h_fn += h_fn
    
    s_f1_list.append(s_f1); h_f1_list.append(h_f1)
    delta = h_f1 - s_f1
    delta_str = f"+{delta*100:.2f}%" if delta >= 0 else f"{delta*100:.2f}%"
    per_class_table.append((cls, s_f1, h_f1, delta_str, h_tp, h_fp, h_fn))

s_mic_p = tot_s_tp / (tot_s_tp + tot_s_fp) if (tot_s_tp + tot_s_fp) > 0 else 0.0
s_mic_r = tot_s_tp / (tot_s_tp + tot_s_fn) if (tot_s_tp + tot_s_fn) > 0 else 0.0
s_mic_f1 = (2 * s_mic_p * s_mic_r) / (s_mic_p + s_mic_r) if (s_mic_p + s_mic_r) > 0 else 0.0
s_mac_f1 = float(np.mean(s_f1_list)) if s_f1_list else 0.0

h_mic_p = tot_h_tp / (tot_h_tp + tot_h_fp) if (tot_h_tp + tot_h_fp) > 0 else 0.0
h_mic_r = tot_h_tp / (tot_h_tp + tot_h_fn) if (tot_h_tp + tot_h_fn) > 0 else 0.0
h_mic_f1 = (2 * h_mic_p * h_mic_r) / (h_mic_p + h_mic_r) if (h_mic_p + h_mic_r) > 0 else 0.0
h_mac_f1 = float(np.mean(h_f1_list)) if h_f1_list else 0.0

print("=" * 114)
print("🏆 VIEW 1: CANDIDATE-LEVEL ARBITRATION BENCHMARK (Decision Accuracy on Proposed Mentions)")
print("   Evaluates decision accuracy on candidate mentions (SpanNER Proposer -> ACE Dynamic Guideline Arbitration)")
print("=" * 114)
print(f"{'Class Name':<48} | {'SpanNER F1':<10} | {'ACE Gen F1':<10} | {'Delta':<8} | {'TP':<5} | {'FP':<5} | {'FN':<5}")
print("-" * 114)
for cls, s_f1, h_f1, d_str, tp, fp, fn in per_class_table:
    print(f"{cls:<48} | {s_f1*100:8.2f}% | {h_f1*100:8.2f}% | {d_str:<8} | {tp:<5} | {fp:<5} | {fn:<5}")
print("-" * 114)
print(f"Candidate Micro Precision : {h_mic_p*100:.2f}%  (SpanNER: {s_mic_p*100:.2f}%, Gain: +{(h_mic_p-s_mic_p)*100:.2f}%)")
print(f"Candidate Micro Recall    : {h_mic_r*100:.2f}%  (SpanNER: {s_mic_r*100:.2f}%, Gain: +{(h_mic_r-s_mic_r)*100:.2f}%)")
print(f"Candidate Micro F1-Score  : {h_mic_f1*100:.2f}%  (SpanNER: {s_mic_f1*100:.2f}%, Gain: +{(h_mic_f1-s_mic_f1)*100:.2f}%)")
print(f"Macro F1-Score (Per-Class): {h_mac_f1*100:.2f}%  (SpanNER: {s_mac_f1*100:.2f}%, Gain: +{(h_mac_f1-s_mac_f1)*100:.2f}%)")
print("=" * 114)

# ------------------------------------------------------------------------------
# 2. VIEW 2: Global End-to-End Dataset Benchmark (Academic Formulations)
# ------------------------------------------------------------------------------
spanner_m = [evaluate_doc_4_formulations(spanner_baseline_test_preds[i], all_test_gts[i], use_new_labels=USE_NEW_LABELS) for i in range(len(test_subset))]
hybrid_m = [evaluate_doc_4_formulations(all_test_preds[i], all_test_gts[i], use_new_labels=USE_NEW_LABELS) for i in range(len(test_subset))]

spanner_results = aggregate_4_formulations(spanner_m)
hybrid_results = aggregate_4_formulations(hybrid_m)

summary_box = format_4_formulations_summary_box(
    spanner_results=spanner_results,
    hybrid_results=hybrid_results,
    active_metrics=["strict", "partial", "mention"],
    system_name="SpanNER + ACE Curated Generator"
)
print("\n" + summary_box)

total_gold_entities = sum(len(gts) for gts in all_test_gts)
covered_gold_entities = hybrid_results["mention"]["tp"]
missed_proposals = total_gold_entities - covered_gold_entities

print("\n" + "=" * 114)
print("💡 ARCHITECTURAL PERFORMANCE ANALYSIS:")
print(f"• Phase 2 (Candidate Arbitration) : {h_mic_f1*100:.2f}% Micro F1 on proposed candidates.")
print(f"• Phase 1 (Span Proposal Ceiling) : Evaluated across all {total_gold_entities} gold entities in test set.")
print(f"  Base SpanNER proposed spans covering {covered_gold_entities} entities, leaving {missed_proposals} entities unproposed.")
print(f"  This retrieval ceiling produces a Global Recall of {hybrid_results['mention']['r']*100:.2f}% and Global Micro F1 of {hybrid_results['mention']['f1']*100:.2f}%.")
print("=" * 114)


In [ ]:
# ==============================================================================
# [CELL 11]: Phase 5 - Export Full Run Reports, JSON Metrics & Abstract Logs
# ==============================================================================
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder_name = f"run_{timestamp_str}_bertized_ace_16k"
base_export_dir = os.path.join(OUTPUT_DIR, "runs", run_folder_name)
abstracts_export_dir = os.path.join(base_export_dir, "abstracts")

os.makedirs(abstracts_export_dir, exist_ok=True)

print("=" * 105)
print(f"  💾 EXPORTING BERTIZED ACE RUN REPORTS & ABSTRACT AUDIT LOGS")
print("=" * 105)
print(f"  • Output Directory : {base_export_dir}")
print(f"  • Total Abstracts  : {len(test_subset)}")

# 1. Save Per-Abstract TXT Audit Files
for doc_idx, (txt_p, ann_p, text, gt_ents, doc_id) in enumerate(test_subset):
    preds = all_test_preds[doc_idx]
    lines = [
        "=" * 100,
        f"📄 ABSTRACT AUDIT REPORT: [DOC_ID: {doc_id}]",
        f"Architecture: BERTized ACE (Generator_v2 + Curated Playbook, 16k tokens)",
        f"Timestamp   : {timestamp_str}",
        "=" * 100,
        "",
        "1. RAW ABSTRACT TEXT & GROUND TRUTH ENTITIES",
        "-" * 100,
        text.strip(),
        "",
        f"Ground Truth Entities ({len(gt_ents)} entities):"
    ]
    for i, g in enumerate(gt_ents, 1):
        g_txt = g["text"]
        lines.append(f"  • [T{i}] ({g['start_char']}, {g['end_char']}): '{g_txt}' -> {g['label']}")
    lines.extend([
        "",
        "2. AUDIT MATRIX (Candidate Spans -> Generator Arbitration)",
        "-" * 100,
        f"{'Span Text':<30} | {'Char Range':<12} | {'Ground Truth':<25} | {'SpanNER Label':<25} | {'Final Label':<25} | Outcome",
        "-" * 130
    ])
    for p in preds:
        st_c, end_c = p["start_char"], p["end_char"]
        gt_lbl = match_gt_label(p["span_text"], st_c, end_c, gt_ents)
        outcome = classify_8_outcome_str(gt_label=gt_lbl, spanner_label=p["spanner_label"], final_label=p["label"], escalated=True)
        lines.append(f"{p['span_text']:<30} | {st_c:>5}-{end_c:<5} | {gt_lbl:<25} | {p['spanner_label']:<25} | {p['label']:<25} | {outcome}")
        
    doc_out = os.path.join(abstracts_export_dir, f"{doc_id}.txt")
    with open(doc_out, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

# 2. Save run_summary_metrics.json
metrics_json = {
    "architecture": "BERTized_ACE_Pipeline",
    "generator_prompt": "ACE_Generator_v2.txt",
    "reflector_prompt": "ACE_Reflector_v2.txt",
    "curator_prompt": "ACE_Curator_v2.txt",
    "timestamp": timestamp_str,
    "model_name": MODEL_NAME,
    "backend": BACKEND,
    "provider": PROVIDER,
    "max_tokens": MAX_TOKENS,
    "total_test_abstracts": len(test_subset),
    "candidate_arbitration": {
        "spanner_micro_f1": s_mic_f1,
        "ace_micro_f1": h_mic_f1,
        "delta": h_mic_f1 - s_mic_f1,
        "spanner_macro_f1": s_mac_f1,
        "ace_macro_f1": h_mac_f1
    },
    "metrics_3_formulations": {
        "strict_conll": {"spanner": spanner_results["strict"], "ace": hybrid_results["strict"]},
        "partial_muc7": {"spanner": spanner_results["partial"], "ace": hybrid_results["partial"]},
        "candidate_mention": {"spanner": spanner_results["mention"], "ace": hybrid_results["mention"]}
    }
}

with open(os.path.join(base_export_dir, "run_summary_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics_json, f, indent=4)

# 3. Save run_summary_report.txt
report_lines = [
    "=" * 100,
    "📊 BERTIZED ACE EXPERIMENT RUN SUMMARY REPORT",
    f"Timestamp: {timestamp_str} | Total Test Abstracts: {len(test_subset)}",
    f"Provider : {PROVIDER} ({API_BASE_URL}) | Model: {MODEL_NAME} | Max Tokens: {MAX_TOKENS}",
    "=" * 100,
    "",
    "1. CANDIDATE ARBITRATION BENCHMARK (View 1)",
    "-" * 100,
    f"  • Candidate Micro F1-Score : {h_mic_p*100:.2f}% / {h_mic_r*100:.2f}% / {h_mic_f1*100:.2f}% (SpanNER: {s_mic_f1*100:.2f}%, Gain: +{(h_mic_f1-s_mic_f1)*100:.2f}%)",
    f"  • Macro F1-Score (Classes) : {h_mac_f1*100:.2f}% (SpanNER: {s_mac_f1*100:.2f}%, Gain: +{(h_mac_f1-s_mac_f1)*100:.2f}%)",
    "",
    "2. GLOBAL END-TO-END DATASET BENCHMARK (View 2)",
    "-" * 100,
    f"{'Formulation':<25} | {'SpanNER Baseline (P/R/F1)':<28} | {'SpanNER + ACE Gen (P/R/F1)':<28} | Gain / Delta",
    "-" * 105,
    f"{'Strict CoNLL (1)':<25} | {spanner_results['strict']['p']*100:>5.1f}% / {spanner_results['strict']['r']*100:>5.1f}% / {spanner_results['strict']['f1']*100:>5.1f}% | {hybrid_results['strict']['p']*100:>5.1f}% / {hybrid_results['strict']['r']*100:>5.1f}% / {hybrid_results['strict']['f1']*100:>5.1f}% | {(hybrid_results['strict']['f1'] - spanner_results['strict']['f1'])*100:>+6.2f}% F1",
    f"{'Partial MUC-7 (3)':<25} | {spanner_results['partial']['p']*100:>5.1f}% / {spanner_results['partial']['r']*100:>5.1f}% / {spanner_results['partial']['f1']*100:>5.1f}% | {hybrid_results['partial']['p']*100:>5.1f}% / {hybrid_results['partial']['r']*100:>5.1f}% / {hybrid_results['partial']['f1']*100:>5.1f}% | {(hybrid_results['partial']['f1'] - spanner_results['partial']['f1'])*100:>+6.2f}% F1",
    f"{'Candidate Mention (4)':<25} | {spanner_results['mention']['p']*100:>5.1f}% / {spanner_results['mention']['r']*100:>5.1f}% / {spanner_results['mention']['f1']*100:>5.1f}% | {hybrid_results['mention']['p']*100:>5.1f}% / {hybrid_results['mention']['r']*100:>5.1f}% / {hybrid_results['mention']['f1']*100:>5.1f}% | {(hybrid_results['mention']['f1'] - spanner_results['mention']['f1'])*100:>+6.2f}% F1",
    "=" * 105
]

with open(os.path.join(base_export_dir, "run_summary_report.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print(f"✅ Successfully exported all run reports, metrics, and abstract logs to: {base_export_dir}")
